In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, confusion_matrix, classification_report
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import shap

In [ ]:
pd.set_option("display.max_columns", None)

# Import and Prepare Data

In [ ]:
# Load csv from file location, if the data was downloaded using the program provided then leave the location as is
df = pd.read_csv("data/accepted_2007_to_2018Q4.csv", low_memory=False)

In [ ]:
# Drop any rows with no date
df = df[df["issue_d"].notna()]

# Filter dates between 2012 and 2017
df["issue_d"] = pd.to_datetime(df["issue_d"])
df1 = df[(df["issue_d"].dt.year >= 2012) & (df["issue_d"].dt.year <= 2017)]

# Filter to rows where loan outcome is known and change the target varibale ie loan outcome to binary under a new tarfet column
df1 = df1[df1["loan_status"].isin(["Default", "Charged Off", "Fully Paid"])]
df1["target"] = df1["loan_status"].isin(["Charged Off", "Default"]).astype(int)

In [ ]:
# Define feature categories

credit_profile = ["fico_range_low", "fico_range_high", "inq_last_6mths", "delinq_2yrs", "open_acc", "total_acc", "pub_rec",
                  "pub_rec_bankruptcies", "revol_util", "revol_bal", "mort_acc", "acc_now_delinq"]

loan_characteristics = ["loan_amnt", "term", "int_rate", "installment", "sub_grade", "purpose", "application_type",
                       "initial_list_status"]

employment_and_income = ["annual_inc", "emp_length", "verification_status", "dti"]

borrower_stability = ["home_ownership", "addr_state"]

timing = ["issue_d", "earliest_cr_line"]

target = ["loan_status", "target"]

model_features = (credit_profile + loan_characteristics + employment_and_income + borrower_stability + timing + target)

model_features

In [ ]:
# Create a dataframe with only the feature variables
df_feat = df1[model_features]

In [ ]:
# Create a column to measure credit history length then drop issue date and earliest credit line
df_feat["earliest_cr_line"] = pd.to_datetime(df["earliest_cr_line"])
df_feat["cr_length_hist"] = ((df_feat["issue_d"] - df_feat["earliest_cr_line"]).dt.days)
df_feat["cr_length_hist"] = df_feat["cr_length_hist"]/365.25
df_feat["cr_length_hist"]

# Drop issue date and earliest credit line column

df_feat = df_feat.drop(columns = ["earliest_cr_line", "issue_d"])

In [ ]:
# Make sure all columns are suitable for encoding particularly dates
df_feat["term"] = df_feat["term"].str[1:3]

# Sort NA rows

# Drop the 1 na row
df_feat = df_feat[df_feat["inq_last_6mths"].notna()]

# NA in both revol util and mort acc means no history therefore fill with 0
df_feat["revol_util"] = df_feat["revol_util"].fillna(0)
df_feat["mort_acc"] = df_feat["mort_acc"].fillna(0)

# DTI only has 240 missing rows so drop them as its a small proportion
df_feat = df_feat[df_feat["dti"].notna()]

# For missing employment length fill with Unknown
df_feat["emp_length"] = df_feat["emp_length"].fillna("Unknown")

In [ ]:
# Change term to integer and also change application_type and initial_list_status to binary encoding as they have 2 unique values
df_feat["term"] = df_feat["term"].astype(int)
df_feat["application_type"] = (df_feat["application_type"] == "Individual").astype(int)
df_feat["initial_list_status"] = (df_feat["initial_list_status"] == "w").astype(int)

df_feat.dtypes

In [ ]:
# Begin Encoding each column that isnt a float or integer

# Ordinal mapping for sub-grade and employment length
employment_len_map = {"Unknown": -1, "< 1 year": 0, "1 year": 1, "2 years": 2, "3 years": 3, "4 years": 4, "5 years": 5,
                         "6 years": 6, "7 years": 7, "8 years": 8, "9 years": 9, "10+ years": 10}

grades = ["A", "B", "C", "D", "E", "F", "G"]
sub_grade_map = {f"{g}{n}": i*5 + n for i, g in enumerate(grades) for n in range(1, 6)}

# Apply ordinal maps
df_feat["emp_length"] = df_feat["emp_length"].map(employment_len_map)
df_feat["sub_grade"] = df_feat["sub_grade"].map(sub_grade_map)

In [ ]:
# Apply one-hot encoding for remaining object columns apart from addr_state

df_feat = pd.get_dummies(df_feat, columns = ["purpose", "application_type", "verification_status", "home_ownership"], drop_first = True, dtype = int)

In [ ]:
df_feat_final = df_feat.drop(columns = ["loan_status", "target"])

# Train and Test Splits

In [ ]:
X = df_feat_final
y = df_feat["target"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 5, stratify = y)

In [ ]:
# Frequency encode addr_state now data has been split into training and test split to ensure correct proportions

state_freq_map = X_train["addr_state"].value_counts(normalize=True)

X_train["addr_state"] = X_train["addr_state"].map(state_freq_map)
X_test["addr_state"] = X_test["addr_state"].map(state_freq_map).fillna(0)

## Logistic Regression

In [ ]:
# Create scalar and scale features
scalar = StandardScaler()
X_train_scaled = scalar.fit_transform(X_train)
X_test_scaled = scalar.transform(X_test)

# Fit Logistic Regression model
log_reg = LogisticRegression(class_weight = "balanced", max_iter = 1000)
log_reg.fit(X_train_scaled, y_train)

In [ ]:
# Calculate ROC and Average Precision Score for Logistic Regression

y_pred_proba_lr = log_reg.predict_proba(X_test_scaled)[:, 1]

lr_roc_auc = roc_auc_score(y_test, y_pred_proba_lr)
lr_pr_auc = average_precision_score(y_test, y_pred_proba_lr)

print(f"Logistic Regression — ROC-AUC: {lr_roc_auc:.4f}")
print(f"Logistic Regression — PR-AUC: {lr_pr_auc:.4f}")

## XGBoost

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    scale_pos_weight = scale_pos_weight,
    eval_metric = "logloss",
    random_state=5
)

xgb_model.fit(X_train, y_train)

In [ ]:
y_pred_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]


xgb_roc_auc = roc_auc_score(y_test, y_pred_proba_xgb)
xgb_pr_auc = average_precision_score(y_test, y_pred_proba_xgb)

print(f"Extreme Gradient Booster — ROC-AUC: {xgb_roc_auc:.4f}")
print(f"Extreme Gradient Booster — PR-AUC: {xgb_pr_auc:.4f}")

## LightGBM 

In [ ]:
lgbm_model = LGBMClassifier(
    scale_pos_weight = scale_pos_weight,
    random_state = 5
)

lgbm_model.fit(X_train, y_train)

In [ ]:
y_pred_proba_lgbm = lgbm_model.predict_proba(X_test)[:, 1]

lgbm_roc_auc = roc_auc_score(y_test, y_pred_proba_lgbm)
lgbm_pr_auc = average_precision_score(y_test, y_pred_proba_lgbm)

print(f"LightGBM — ROC-AUC: {lgbm_roc_auc:.4f}")
print(f"LightGBM — PR-AUC: {lgbm_pr_auc:.4f}")

In [ ]:
model_score_table = [{"Model": "Logistic Regression", "ROC AUC Score": lr_roc_auc, "PR AUC Score": lr_pr_auc},
                     {"Model": "Extreme Gradient Boosting", "ROC AUC Score": xgb_roc_auc, "PR AUC Score": xgb_pr_auc},
                    {"Model": "Light GBM", "ROC AUC Score": lgbm_roc_auc, "PR AUC Score": lgbm_pr_auc}]

model_score_table_df = pd.DataFrame(model_score_table)
model_score_table_df.round(4)

As the XGBoosting Model score both the best ROC AUC and PR AUC score I will use that model for the remainder of the project and apply the model to the real world scenarios of classifying loans.

# Setting the Threshold and Threshold Analysis (XGBoosting)

Begin by plotting a Precision Recall curve and then identify a suitable threshold for the scenario - loan classification for this project. As a result the threshold will be below 0.5 as a threshold of 0.5 implies the cost of false positives (flagging a good loan) is the same as a false negative (missing a bad loan). As we know the cost of a false negative is a lot higher as missing a bad loan that will default will result in debt collection and higher costs whereas flagging good loans only requires a manual check which is way less costly. Therefore the threshold will be below 0.5 as we would rather have false positives than false negatives.

In [ ]:
precisions, recalls, thresholds = precision_recall_curve(y_test, y_pred_proba_xgb)

plt.plot(thresholds, precisions[:-1], label="Precision")
plt.plot(thresholds, recalls[:-1], label="Recall")
plt.title("Precision Recall Curve")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.legend()
plt.show()

In [ ]:
threshold_table = pd.DataFrame({
    "threshold": thresholds,
    "precision": precisions[:-1],
    "recall": recalls[:-1]
})

# I want to catch 85% of defaults therefore set the recall cutoff point to 0.85
display(threshold_table[threshold_table["recall"] >= 0.85].tail(1))

# Set the threshold and apply the threshold
threshold = 0.376
y_pred_at_threshold = (y_pred_proba_xgb >= threshold).astype(int)

The precision recall curves shows that at a low threshold the model is flagging almost all the loans as defaults seen by the recall score >0.8 however the precision is very low indicating a large proportion of false positives. On the other end with a high threshold the model flags a small amount of loans as defaults therefore missing a large proportion of defaults seen by the low recall. However, when it does flag a loan as a default it is often correct seen by the high precision score. 

For my model I want to catch 85% of the defaults therefore have a recall rate of 0.85 which when checking the raw numbers gives a threshold rate of 0.376 which looks accurate with the graph. 

The next step is to apply the threshold to the test set. The model currently asigns a level of risk between 0 and 1 for each loan by applying the threshold any risk above that threshold will be flagged as a "risky" loan whereas any score below the threshold is flagged as a "safe" loan. 

## Threshold Analysis

In [ ]:
# Plot confusion matrix
total_loans = len(y_pred_at_threshold)
threshold_confusion_matrix = confusion_matrix(y_test, y_pred_at_threshold)
raw_threshold_confusion_matrix_df = pd.DataFrame([{"": "True: 0", "Predicted: 0": threshold_confusion_matrix[0,0], "Predicted: 1": threshold_confusion_matrix[0,1]},
                                {"": "True: 1", "Predicted: 0": threshold_confusion_matrix[1,0], "Predicted: 1": threshold_confusion_matrix[1,1]}])

proportional_threshold_confusion_matrix_df = pd.DataFrame([{"": "True: 0", "Predicted: 0": threshold_confusion_matrix[0,0]/total_loans*100, "Predicted: 1": threshold_confusion_matrix[0,1]/total_loans*100},
                                {"": "True: 1", "Predicted: 0": threshold_confusion_matrix[1,0]/total_loans*100, "Predicted: 1": threshold_confusion_matrix[1,1]/total_loans*100}])
proportional_threshold_confusion_matrix_df = (proportional_threshold_confusion_matrix_df.style.format({
    "Predicted: 0": "{:.2f}%",
    "Predicted: 1": "{:.2f}%"
}))



print("Raw Confusion Matrix")
display(raw_threshold_confusion_matrix_df)
print("Proportional Confusion Matrix")
display(proportional_threshold_confusion_matrix_df)
print(f"Total Loans: {total_loans}")

The confusion matrix shows the model is acheiving a high recall rate for the dault class which was the aim of the model and is only missing 7500 defaulted loans. As expected there a large amount of false positives with 112,000 or 45% of all loans. While it may be time consuming to manually check each of the flagged loans the overall cost is likely much lower than missing a risky loan. This was also taken into acount when setting the threshold, therefore, the confusion matrix shows the model is working as intended. 

If the threshold was raised this would lead to a higher precision rate which would result in less false positives, however, the false neagtive rate would also rise leading to more risky loans being missed.

In [ ]:
threshold_classification_report = classification_report(y_test, y_pred_at_threshold)
print(threshold_classification_report)

The classification report shows the threshold was set correctly as the intention was to catch 85% of "risky" loans which is correctly reflected by a recall rate of 0.85 for defaulted loans. The recall rate for "safe" loans (0.44) and a low precision rate when predicting "risky" loans (0.28). These scores are a result of using a low threshold rate resulting in a large proportion of false positives which leads to many "safe" loans predicted as "risky" loans leading to a low recall rate and large number of "risky" loans are actually safe but are flagged due to the low threshold reuslting in the low precision rate.

## SHAP Values

In [ ]:
# Set explainer and calculate shap values 
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

In [ ]:
# Plot summary plot

shap.summary_plot(shap_values, X_test)

The SHAP summary plot shows the model's predictions are primarily driven by credit related features such as the sub-grade, interest rate, debt-to-income ration (dti), loan amount and FICO score. Higher sub-grades, longer term loans, higher interest rates, lower FICO scores and higher debt-to-income ratios all lead to the model predicting towards a default. This is how I expected the model to behave and the SHAP values indicate the model has learnt relationships that align with established credit-risk behavior and using the features as expected. 

Looking at the scores we can see that a low subgrade (ie a good grade) has the largest impact on the prediction with the remaining predictors all having approxiamately the same impact on the perdiciton with a few such as home onwership status, verification status source having minimal impact on the prediction. Interestingly when looking at the term feature there were only 2 unique values either 36 months or 60 months indicated by the blue and red key. A 36 month term has little impact on the prediction wheras a longer term of 60 months has a much larger impact on the perdiction pushing it towards a default prediction. This is likely due to the fact longer terms are often seen as a high risk signal resulting in a noticeable sway towards a default prediction, whereas a 36 month term is seen as typical in the dataset therefore having little impact on the prediction.

In [ ]:
# Highest risk and lowest risk loan predicted by the model
high_risk_idx = y_pred_proba_xgb.argmax()
low_risk_idx = y_pred_proba_xgb.argmin()

# For borderline prediciton pick the loan closest to the threshold 
borderline_idx = (abs(y_pred_proba_xgb - threshold)).argmin()

In [ ]:
# Plot waterfall for high risk loan

shap.plots.waterfall(shap.Explanation(
    values=shap_values[high_risk_idx],
    base_values=explainer.expected_value,
    data=X_test.iloc[high_risk_idx],
    feature_names=X_test.columns.tolist()
))

The SHAP values for the loan with the highest predicted risk reflects the summary plot as the loan has the 3rd from worst sub_grade which is the feature that has the largest impact on the prediction which was expected from looking at the summary plot. The interest rate had a similar impact on the prediction as the sub grade, for this loan the interest rate was 29.67% which is 3.5 standard deviations above the mean and overall a very high interest rate so it's unsurprising this significantly pushed the prediction towards a default. The loan was also being used for a small business which is often seen as a risky loan due to the lack of collateral. Furthermore, the loan has a term of 60 months which combined with the high interest rate reuslts in the amount needing to be paid off being much higher than the loan amount reuslting in a high chance of defaulting. Overall, this loan has poor features that we expect to result in the loan being classified as a high risk loan when comparing to the SHAP summary graph. 

In [ ]:
# Plot waterfall for low risk loan

shap.plots.waterfall(shap.Explanation(
    values=shap_values[low_risk_idx],
    base_values=explainer.expected_value,
    data=X_test.iloc[low_risk_idx],
    feature_names=X_test.columns.tolist()
))

This SHAP waterfall chart explains why the model assigns this borrower the lowest predicted risk of default. The model’s baseline prediction is close to zero, and each feature pushes the score further downward. The largest negative contributions come from a very strong sub‑grade, a high FICO score, low revolving utilisation, and a high annual income, all of which signal strong creditworthiness. Additional stabilising factors—such as a long credit history, low inquiry activity, stable employment length, and multiple established accounts—further reduce the predicted risk. Together, these features create a cumulative SHAP impact that drives the model’s output far below the average prediction, indicating that this borrower exhibits exceptionally low default risk across multiple dimensions of their financial profile.

In [ ]:
# Plot waterfall for borderline loan

shap.plots.waterfall(shap.Explanation(
    values=shap_values[borderline_idx],
    base_values=explainer.expected_value,
    data=X_test.iloc[borderline_idx],
    feature_names=X_test.columns.tolist()
))

This SHAP waterfall chart explains why the model predicted this borrower’s risk level very close to the classification threshold. The baseline prediction is near zero, and the final output remains only slightly negative, indicating a finely balanced risk profile. Several features push the prediction upward toward default—most notably the borrower’s interest rate, lower FICO score, and mortgage home‑ownership status, all of which signal moderate financial risk. However, these upward pressures are almost fully offset by strong downward contributions from a favourable sub‑grade, a shorter loan term, a reasonable debt‑to‑income ratio, no recent credit inquiries, and a stable annual income. The result is a prediction where positive and negative SHAP impacts nearly cancel each other out, placing this borrower right on the boundary between low and elevated risk. This chart highlights how the model weighs competing signals when assessing borderline cases.